# PowerFit in your browser

This notebook runs PowerFit's entirely client-side via [Pyodide](https://pyodide.org)/[JupyterLite](https://jupyterlite.readthedocs.io).

Run all cells (top toolbar, or Shift+Enter through each one). First run takes 15-30s while Pyodide and the PowerFit wasm wheel load.

In [ ]:
try:
    import piplite

    # pygments is required by rich.traceback (imported by powerfit_em.powerfit) but
    # Pyodide's curated "rich" build doesn't pull it in automatically
    await piplite.install(["pygments", "powerfit_em"])
except ModuleNotFoundError:
    pass  # not running under Pyodide, assume powerfit_em is already installed

In [ ]:
from powerfit_em.powerfit_rs import cargo_version

cargo_version()

## Data

Target density map and template structure are fetched from [haddocking/powerfit-tutorial](https://github.com/haddocking/powerfit-tutorial).

In [ ]:
from pyodide.http import pyfetch

base = "https://raw.githubusercontent.com/haddocking/powerfit-tutorial/master"
target_volume_fn = "ribosome-KsgA.map"
template_structure_fn = "KsgA.pdb"

for name in (target_volume_fn, template_structure_fn):
    response = await pyfetch(f"{base}/{name}")
    with open(name, "wb") as f:
        f.write(await response.bytes())

In [ ]:
from powerfit_em.powerfit import powerfit

output_dir = "output"

with open(target_volume_fn, "rb") as target, open(template_structure_fn, "rb") as template:
    powerfit(
        target_volume=target,
        template_structure=template,
        resolution=13,
        angle=20,
        rust=True,
        nproc=2,
        directory=output_dir,
    )

In [ ]:
from pathlib import Path

list(Path(output_dir).glob("*"))

In [ ]:
print(Path(output_dir, "solutions.out").read_text())